# SuperNormal-RS — CPU-Only Prebuild Notebook (CUDA + WGPU)

Use this notebook on a **free CPU-only Google Colab runtime** (`Runtime -> Change runtime type -> CPU`) so you don't spend any A100 / L4 / G4 GPU compute units while compiling Rust.

It will:
1. Mount your Google Drive.
2. Verify or install the **CUDA Toolkit** (headers + NVRTC libraries needed to compile `burn/cuda` & `cubecl-cuda`).
3. Install the **Rust toolchain** (`rustup`).
4. Clone your `supernormal-rs` repo and compile the Linux `release` binary with **CUDA + WGPU** support.
5. Save the prebuilt binary to Google Drive (`/content/drive/MyDrive/supernormal_colab/bin/supernormal`) for instant reuse in a GPU notebook.

> **Before running:** Make sure you have pushed your latest `main` branch (`git push origin main`) so Colab clones the CUDA-enabled commit.

In [ ]:
# === Configuration ===
REPO_URL = "https://github.com/pavelf200205/supernormal-rs.git"
BRANCH = "main"  # Change to "snsr" if you merge CUDA into snsr later
DRIVE_DIR = "/content/drive/MyDrive/supernormal_colab"
BINARY_NAME = f"supernormal_{BRANCH}_cuda"

# Fast release build (~3-5 min on CPU, identical GPU SPIR-V/PTX kernel speed).
# Set FULL_LTO = True only if you want full cross-crate host LTO (~12-15 min on CPU).
FULL_LTO = False

print(f"Repo:   {REPO_URL} (branch: {BRANCH})")
print(f"Output: {DRIVE_DIR}/bin/{BINARY_NAME}")

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(f"{DRIVE_DIR}/bin", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/datasets", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/experiments", exist_ok=True)
print("Drive directories ready under:", DRIVE_DIR)

In [ ]:
# 2. Check / install system build dependencies + CUDA Toolkit (headers & NVRTC)
%%bash
set -eu

echo "=== OS & glibc info ==="
lsb_release -a 2>/dev/null || cat /etc/os-release
ldd --version | head -n 1

echo "=== Installing base build tools (build-essential, pkg-config, libclang-dev, xz-utils) ==="
apt-get update -qq
apt-get install -y -qq build-essential pkg-config libclang-dev cmake curl git xz-utils > /dev/null

if command -v nvcc >/dev/null 2>&1 && [ -d /usr/local/cuda/include ]; then
    echo "=== CUDA Toolkit already installed on Colab VM ==="
    nvcc --version
else
    echo "=== Installing NVIDIA CUDA 12.x Toolkit (auto-detecting latest version for Ubuntu ${UBUNTU_VER:-}) ==="
    UBUNTU_VER=$(. /etc/os-release && echo "${VERSION_ID//./}")
    wget -q "https://developer.download.nvidia.com/compute/cuda/repos/ubuntu${UBUNTU_VER}/x86_64/cuda-keyring_1.1-1_all.deb" -O /tmp/cuda-keyring.deb
    dpkg -i /tmp/cuda-keyring.deb > /dev/null
    apt-get update -qq
    CUDA_PKG=$(apt-cache pkgnames cuda-nvcc-12- | sort -V | tail -n 1)
    CUDA_VER=${CUDA_PKG#cuda-nvcc-}
    CUDA_DOT=${CUDA_VER//-/.}
    echo "Installing CUDA ${CUDA_DOT} packages..."
    apt-get install -y -qq "cuda-nvcc-${CUDA_VER}" "cuda-cudart-dev-${CUDA_VER}" "cuda-nvrtc-dev-${CUDA_VER}" "cuda-driver-dev-${CUDA_VER}" > /dev/null
    ln -sfn "/usr/local/cuda-${CUDA_DOT}" /usr/local/cuda
    /usr/local/cuda/bin/nvcc --version
fi

echo "=== CUDA libraries present ==="
ls -la /usr/local/cuda/lib64/libnvrtc* /usr/local/cuda/include/cuda.h 2>/dev/null || true

In [ ]:
# 3. Install Rust toolchain (stable)
%%bash
set -eu
if ! command -v cargo >/dev/null 2>&1; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
source "$HOME/.cargo/env"
rustc --version
cargo --version

In [ ]:
# 4. Clone repository (supports both repo root and nested supernormal-rs layout)
import os, subprocess

WORK_DIR = "/content/supernormal-src"
if os.path.exists(WORK_DIR):
    subprocess.run(["rm", "-rf", WORK_DIR], check=True)

subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK_DIR], check=True)

# Detect whether Cargo.toml is at repo root or in supernormal-rs/
if os.path.isfile(os.path.join(WORK_DIR, "Cargo.toml")):
    CRATE_DIR = WORK_DIR
elif os.path.isfile(os.path.join(WORK_DIR, "supernormal-rs", "Cargo.toml")):
    CRATE_DIR = os.path.join(WORK_DIR, "supernormal-rs")
else:
    raise RuntimeError("Could not find Cargo.toml in cloned repository!")

COMMIT_HASH = subprocess.check_output(["git", "-C", WORK_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"Crate directory: {CRATE_DIR}")
print(f"Checked out commit: {COMMIT_HASH} ({BRANCH})")

In [ ]:
# 5. Build SuperNormal-RS (CUDA + WGPU) on CPU
import os, subprocess, time

env = os.environ.copy()
env["PATH"] = f"{os.path.expanduser('~')}/.cargo/bin:/usr/local/cuda/bin:" + env.get("PATH", "")
env["CUDA_PATH"] = "/usr/local/cuda"
env["CUDA_ROOT"] = "/usr/local/cuda"
env["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")

if not FULL_LTO:
    env["CARGO_PROFILE_RELEASE_LTO"] = "false"
    env["CARGO_PROFILE_RELEASE_CODEGEN_UNITS"] = "16"

t0 = time.time()
proc = subprocess.Popen(
    ["cargo", "build", "--release", "--features", "cuda"],
    cwd=CRATE_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
elapsed = time.time() - t0
if ret != 0:
    raise RuntimeError(f"cargo build failed with exit code {ret}")

print(f"\nBuild succeeded in {elapsed/60:.1f} min ({elapsed:.1f}s)!")

In [ ]:
# 6. Persist the prebuilt binary + build metadata to Google Drive
import os, shutil, subprocess, datetime

src_bin = os.path.join(CRATE_DIR, "target", "release", "supernormal")
dst_bin = os.path.join(DRIVE_DIR, "bin", BINARY_NAME)
dst_latest = os.path.join(DRIVE_DIR, "bin", "supernormal")

shutil.copy2(src_bin, dst_bin)
shutil.copy2(src_bin, dst_latest)
os.chmod(dst_bin, 0o755)
os.chmod(dst_latest, 0o755)

size_mb = os.path.getsize(dst_bin) / (1024 * 1024)
glibc_ver = subprocess.check_output(["ldd", "--version"]).decode().splitlines()[0]
nvcc_ver = subprocess.check_output(["/usr/local/cuda/bin/nvcc", "--version"]).decode().strip()

info_path = os.path.join(DRIVE_DIR, "bin", f"{BINARY_NAME}_build_info.txt")
with open(info_path, "w") as f:
    f.write(f"Built at: {datetime.datetime.utcnow().isoformat()}Z\n")
    f.write(f"Repo:     {REPO_URL}\n")
    f.write(f"Branch:   {BRANCH}\n")
    f.write(f"Commit:   {COMMIT_HASH}\n")
    f.write(f"Full LTO: {FULL_LTO}\n")
    f.write(f"Size:     {size_mb:.2f} MB\n")
    f.write(f"glibc:    {glibc_ver}\n")
    f.write(f"NVCC:\n{nvcc_ver}\n")

print(f"Saved binary ({size_mb:.2f} MB) to:\n  - {dst_bin}\n  - {dst_latest}")
print("Build metadata saved to:", info_path)
print("\nYou can now disconnect this CPU runtime and open colab_run_supernormal_gpu.ipynb on an A100 / L4 / G4 GPU!")